# Sierra Nevada worked example

The single-range walk-through of the mountain-range temperature-sensitivity analysis: the N-year median onset and spring 2 m temperature, the
per-year anomaly maps of both, their range means, and the regression of the onset anomaly on the
spring-temperature anomaly. Reads the dataset store directly (4x coarsened), the ERA5-Land stack
and the hillshade in `data/`.

**Pixel rule.** Pixels with forest cover fraction > 50 % are dropped from every runoff variable right
after the store read, mirroring the pipeline's `fcf_lte_50` filter (`aggregate.FILTERS`) so this
notebook and `mountain_ranges/temperature_sensitivity.ipynb` describe the same population.

Setup cells first (imports, config, AOI, hillshade, the store read + FCF mask, ERA5), then any
figure cell; the last cell assembles the composite figure natively (no slide step).

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import xyzservices as xyz
import cartopy.crs as ccrs
from cartopy import feature as cfeature
from global_snowmelt_runoff_onset.config import Config, Tile
from gsro_analysis import paths, settings
import easysnowdata
import geopandas as gpd
import numpy as np
import pandas as pd
import textwrap
import rasterio
import rioxarray
import matplotlib.patheffects as path_effects

In [ ]:
# Helper function to plot geometries efficiently
def plot_geoms(gdf, ax, color="black", linewidth=1, transform=None, **kwargs):
    """Plot geometries efficiently without geopandas overhead"""
    for geom in gdf.geometry:
        if geom is None:
            continue
        if geom.geom_type in ["Polygon", "MultiPolygon"]:
            # Get exterior coordinates
            if geom.geom_type == "Polygon":
                polys = [geom]
            else:
                polys = list(geom.geoms)
            for poly in polys:
                x, y = poly.exterior.xy
                ax.plot(x, y, color=color, linewidth=linewidth, transform=transform, **kwargs)

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

In [ ]:
global_ds = config.open_runoff_onset_dataset()
global_ds

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
sierra_nevada_gdf = gmba_gdf[gmba_gdf['MapName']=='Sierra Nevada'].to_crs("EPSG:32611")
sierra_nevada_gdf

In [ ]:
sierras_hillshade_da = rioxarray.open_rasterio(paths.DATA / 'global_hillshade_robinson.tif', masked=True, chunks='auto').rio.clip_box(*sierra_nevada_gdf.buffer(50000).total_bounds,crs=sierra_nevada_gdf.crs).squeeze().rio.reproject(sierra_nevada_gdf.crs, resampling=rasterio.enums.Resampling.bilinear)
sierras_hillshade_da

In [ ]:
sierra_nevada_4326_gdf = sierra_nevada_gdf.to_crs("EPSG:4326")

In [ ]:
xmin, ymin, xmax, ymax = sierra_nevada_4326_gdf.total_bounds
import cartopy.crs as ccrs
import cartopy.feature as cfeature
fig = plt.figure(figsize=(6,6))
central_longitude = (xmin + xmax) / 2
central_latitude = (ymin + ymax) / 2
ax = plt.axes(projection=ccrs.Orthographic(central_longitude=central_longitude, central_latitude=central_latitude))
#ax.set_extent([-180, -100, 50, 80], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, zorder=0)
ax.add_feature(cfeature.OCEAN, zorder=0)
#ax.add_feature(cfeature.COASTLINE, zorder=1)
#ax.add_feature(cfeature.BORDERS, zorder=1)
#ax.add_feature(cfeature.LAKES, zorder=1)
#ax.add_feature(cfeature.RIVERS, zorder=1)
ax.add_geometries(sierra_nevada_4326_gdf.geometry, crs=ccrs.PlateCarree(), edgecolor='black', facecolor='none', linewidth=0.3, zorder=2)
from shapely.geometry import box
bbox = box(xmin-1E5, ymin-1E5, xmax+1E5, ymax+1E5)


fig.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_context_map.png', dpi=300, transparent=True)

In [ ]:
# The three variables the figures use are selected BEFORE the read: the yearly temporal_resolution and
# the float64 ancillaries would otherwise double the peak memory of this ~108 M-pixel window (2026-09-03
# memory pass). Then 4x coarsened -> ~6.75 M cells x 13 layers, float32.
sierra_nevada_ds = (global_ds[['runoff_onset', 'runoff_onset_median', 'runoff_onset_mad']]
                    .sel(latitude=slice(42, 34), longitude=slice(-124, -117))
                    .coarsen(latitude=4, longitude=4, boundary='trim').mean()
                    .astype('float32').compute())
sierra_nevada_ds

In [ ]:
sierra_nevada_ds['runoff_onset_anomaly'] = sierra_nevada_ds['runoff_onset'] - sierra_nevada_ds['runoff_onset_median']
sierra_nevada_ds

In [ ]:
sierra_nevada_ds = sierra_nevada_ds.rio.reproject("EPSG:32611")
cartopy_crs = ccrs.UTM(11, southern_hemisphere=False)
sierra_nevada_ds

In [ ]:
# Forest-cover filter — the pipeline's `fcf_lte_50` rule applied on this notebook's 4x-coarsened
# UTM grid (FCF resampled bilinearly onto it): the same rule, not the identical pixel mask.
# the same PROBA-V LC100 tree-cover-fraction GeoTIFF the ancillary build reads (settings.FOREST_COVER_FRACTION_URL,
# a public mirror; the Zenodo original fails intermittently on ranged reads)
fcf_da = (rioxarray.open_rasterio(settings.FOREST_COVER_FRACTION_URL, mask_and_scale=True, chunks=True).squeeze()
          .rio.clip_box(*sierra_nevada_gdf.to_crs('EPSG:4326').total_bounds, crs='EPSG:4326').compute())
fcf_match_da = fcf_da.rio.reproject_match(sierra_nevada_ds['runoff_onset_anomaly'], resampling=rasterio.enums.Resampling.bilinear)
sierra_nevada_ds = sierra_nevada_ds.where(fcf_match_da <= 50)
print(f"{float((fcf_match_da <= 50).mean()):.0%} of the coarsened grid has FCF <= 50 %")
sierra_nevada_ds

In [ ]:
f,axs=plt.subplots(1,2,figsize=(6,6),subplot_kw={'projection':cartopy_crs},layout='constrained',sharey=True,sharex=True)

sierra_nevada_ds['runoff_onset_median'].plot.imshow(ax=axs[0],vmin=110,vmax=250,transform=cartopy_crs,cbar_kwargs={'label': 'DOWY','orientation': 'horizontal'},zorder=1)
sierra_nevada_ds['runoff_onset_mad'].plot.imshow(ax=axs[1],vmin=0,vmax=30,cmap='Reds',transform=cartopy_crs,cbar_kwargs={'label': 'Days','orientation': 'horizontal'},zorder=1)


for ax in axs:
    sierras_hillshade_da.plot.imshow(cmap='grey',ax=ax,transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False,zorder=0)
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
    gl.right_labels = False
    gl.top_labels = False
    ax.set_title('')
    sierra_nevada_gdf.boundary.plot(color='black',ax=ax,transform=cartopy_crs)
    ax.set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

    #ax.set_facecolor('darkgray')    

gl.left_labels = False

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_runoff_onset_median_and_mad.png', dpi=300, bbox_inches='tight',pad_inches=0.0)

In [ ]:
f,ax=plt.subplots(figsize=(5,5),subplot_kw={'projection':cartopy_crs},layout='constrained',sharey=True,sharex=True)
sierras_hillshade_da.plot.imshow(cmap='grey',ax=ax,transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)
sierra_nevada_ds['runoff_onset_median'].plot.imshow(ax=ax,vmin=110,vmax=250,cmap='viridis',transform=cartopy_crs,cbar_kwargs={'label': 'Median runoff onset [DOWY]','pad':0.01})
#ax.set_facecolor('darkgray')
ax.set_aspect('equal')

gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='lightgray', alpha=0.9, linestyle='--')
gl.right_labels = False
gl.top_labels = False
#ax.set_title('WY2015-2024 median runoff onset')
ax.set_title('')
sierra_nevada_gdf.boundary.plot(color='black',ax=ax,transform=cartopy_crs)
ax.set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_snowmelt_median_runoff_onset.png', dpi=300, bbox_inches='tight')

In [ ]:
# now create a figure where the top row is the median and the bottom row is the anomaly
n_wy = len(sierra_nevada_ds.water_year)   # one column per water year (from the data, never a literal)
f, axs = plt.subplots(2, n_wy, figsize=(1.2 * n_wy, 4), subplot_kw={'projection': cartopy_crs},layout='constrained',dpi=300)

for i, water_year in enumerate(sierra_nevada_ds.water_year.values):

    if water_year == sierra_nevada_ds.water_year.values[-1]:  # last column gets the colorbar
        colorbar_flag = True
        cbar_dowy = {"label":"Annual runoff onset\n[DOWY]"}
        cbar_anomaly = {"label":"Runoff onset anomaly\n[days]"}
    else:
        colorbar_flag = False
        cbar_dowy = None
        cbar_anomaly = None

    sierras_hillshade_da.plot.imshow(cmap='grey',ax=axs[0, i],transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)
    sierras_hillshade_da.plot.imshow(cmap='grey',ax=axs[1, i],transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)

    sierra_nevada_ds['runoff_onset'].sel(water_year=water_year).plot.imshow(ax=axs[0, i], vmin=110, vmax=250, transform=cartopy_crs,add_colorbar=colorbar_flag,cbar_kwargs=cbar_dowy)
    sierra_nevada_ds['runoff_onset_anomaly'].sel(water_year=water_year).plot.imshow(ax=axs[1, i], vmin=-30, vmax=30, cmap='RdBu', transform=cartopy_crs,add_colorbar=colorbar_flag,cbar_kwargs=cbar_anomaly)

    axs[0, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
    axs[1, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')

    axs[0, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())
    axs[1, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

    #sierra_nevada_gdf.boundary.plot(color='black',ax=axs[0,i],transform=cartopy_crs)
    #sierra_nevada_gdf.boundary.plot(color='black',ax=axs[1,i],transform=cartopy_crs)
    plot_geoms(sierra_nevada_gdf, axs[0,i], color='black', linewidth=1, transform=cartopy_crs)
    plot_geoms(sierra_nevada_gdf, axs[1,i], color='black', linewidth=1, transform=cartopy_crs)

    axs[0,i].set_title(f"WY{water_year}") # , fontsize=10
    axs[1,i].set_title(f"")
    
    axs[0,i].set_aspect('equal')
    axs[1,i].set_aspect('equal')

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_snowmelt_runoff_onset_and_anomaly.png', dpi=300, bbox_inches='tight')

In [ ]:
mean_runoff_anomaly_da = sierra_nevada_ds['runoff_onset_anomaly'].rio.clip(sierra_nevada_gdf.geometry).mean(dim=['x','y'])
mean_runoff_anomaly_da

In [ ]:
# make a bar chart of the mean anomaly, each bar color should be color coded by RdBu cmap.... -30 darkest red, 30 darkest blue
f,ax=plt.subplots(figsize=(12,2),dpi=300)
my_colormap = plt.get_cmap('RdBu')
norm = plt.Normalize(vmin=-30, vmax=30)
colors = my_colormap(norm(mean_runoff_anomaly_da.values))
mean_runoff_anomaly_da.to_series().plot.bar(ax=ax, color=colors, edgecolor='black')
ax.axhline(0, color='black', linestyle='--')
ax.set_ylabel('Average anomaly\n[days]')
ax.set_xlabel('')
# rotate x labels and remove tick marks. label should be WYXXXX instead of XXXX

ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True,length=0)
ax.set_xticklabels([f'WY{water_year}' for water_year in mean_runoff_anomaly_da.water_year.values], rotation=0)
ax.set_xticklabels([f'' for water_year in mean_runoff_anomaly_da.water_year.values], rotation=0)

# turn off the x axis spines
ax.spines['top'].set_visible(False)

ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# label the top of each bar with the value
for i, v in enumerate(mean_runoff_anomaly_da.values):
    if v < 0:
        ax.text(i, v - 0.8, f'{v:.1f}', ha='center', va='top')
    else:
        ax.text(i, v + 0.5, f'{v:.1f}', ha='center', va='bottom')#, fontsize=10

ax.set_ylim(-40, 20)

# switch the y axis to be on the right

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

f.tight_layout()
f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_snowmelt_runoff_onset_anomaly_barchart.png', dpi=300, bbox_inches='tight')

In [ ]:
from gsro_analysis import era5

era5_land_ds = era5.open_era5_land(config)   # native 0.1 deg grid, latitude north-down
era5_land_ds

In [ ]:
#sierras_spring_temp_avgs_da = era5_land_ds['temperature_2m'].rio.clip_box(*sierra_nevada_gdf.total_bounds, crs=sierra_nevada_gdf.crs).sel(month=['spring_month_1','spring_month_2','spring_month_3']).mean(dim='month').compute()
sierras_spring_temp_avgs_da = era5_land_ds['temperature_2m'].sel(latitude=slice(42, 34), longitude=slice(-124, -117)).sel(month=['spring_month_1','spring_month_2','spring_month_3']).mean(dim='month').compute()
sierras_spring_temp_avgs_da

In [ ]:
# close the era5_land_ds to free up memory
del era5_land_ds

In [ ]:
sierras_spring_temp_avgs_projmatch_da = sierras_spring_temp_avgs_da.rio.reproject_match(sierra_nevada_ds['runoff_onset_anomaly'],resampling=rasterio.enums.Resampling.bilinear)
sierras_spring_temp_avgs_projmatch_da

In [ ]:
sierras_spring_temp_avgs_projmatch_masked_da = sierras_spring_temp_avgs_projmatch_da.where(sierra_nevada_ds['runoff_onset_anomaly'].notnull(), drop=False)
sierras_spring_temp_avgs_projmatch_masked_da

In [ ]:
sierras_spring_temp_10yr_median_da = sierras_spring_temp_avgs_projmatch_masked_da.median(dim='water_year').where(sierras_spring_temp_avgs_projmatch_masked_da.count(dim='water_year')>=3)
sierras_spring_temp_10yr_median_da

In [ ]:
sierras_spring_anomaly_da = sierras_spring_temp_avgs_projmatch_masked_da - sierras_spring_temp_10yr_median_da
sierras_spring_anomaly_da

In [ ]:
sierras_spring_anomaly_full_match_da = sierras_spring_anomaly_da.where(sierra_nevada_ds['runoff_onset_anomaly'].notnull(), drop=False)
sierras_spring_anomaly_full_match_da

In [ ]:
sierras_spring_temp_C_avgs_projmatch_masked_da = sierras_spring_temp_avgs_projmatch_masked_da - 273.15
sierras_spring_temp_C_avgs_projmatch_masked_da

In [ ]:

f,ax=plt.subplots(figsize=(5,5),subplot_kw={'projection':cartopy_crs},layout='constrained',sharey=True,sharex=True)
sierras_hillshade_da.plot.imshow(cmap='grey',ax=ax,transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)

(sierras_spring_temp_10yr_median_da-273.15).plot.imshow(ax=ax,vmin=-5,vmax=10,cmap='plasma',transform=cartopy_crs,cbar_kwargs={'label': 'Spring temperature [°C]','pad':0.01})
#ax.set_facecolor('darkgray')
ax.set_aspect('equal')

gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='lightgray', alpha=0.9, linestyle='--')
gl.right_labels = False
gl.top_labels = False
#ax.set_title('WY2015-2024 median spring temp.')
ax.set_title('')
sierra_nevada_gdf.boundary.plot(color='black',ax=ax,transform=cartopy_crs)
ax.set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_spring_temp_median.png', dpi=300, bbox_inches='tight')

In [ ]:
# now create a figure where the top row is the median and the bottom row is the anomaly
n_wy = len(sierra_nevada_ds.water_year)   # one column per water year (from the data, never a literal)
f, axs = plt.subplots(2, n_wy, figsize=(1.2 * n_wy, 4), subplot_kw={'projection': cartopy_crs},layout='constrained',dpi=300)

for i, water_year in enumerate(sierra_nevada_ds.water_year.values):

    if water_year == sierra_nevada_ds.water_year.values[-1]:  # last column gets the colorbar
        colorbar_flag = True
        cbar_dowy = {"label":"Avg. Spring temp.\n[°C]"}
        cbar_anomaly = {"label":"Spring temp. anomaly\n[°C]"}
    else:
        colorbar_flag = False
        cbar_dowy = None
        cbar_anomaly = None

    sierras_spring_temp_C_avgs_projmatch_masked_da.sel(water_year=water_year).plot.imshow(ax=axs[0, i], vmin=-10,vmax=10,cmap='plasma',transform=cartopy_crs,add_colorbar=colorbar_flag,cbar_kwargs=cbar_dowy) # vmin=110, vmax=250,
    sierras_spring_anomaly_full_match_da.sel(water_year=water_year).plot.imshow(ax=axs[1, i], vmin=-3, vmax=3, cmap='RdBu_r', transform=cartopy_crs,add_colorbar=colorbar_flag,cbar_kwargs=cbar_anomaly)

    axs[0, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
    axs[1, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')

    axs[0, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())
    axs[1, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

    sierra_nevada_gdf.boundary.plot(color='black',ax=axs[0,i],transform=cartopy_crs)
    sierra_nevada_gdf.boundary.plot(color='black',ax=axs[1,i],transform=cartopy_crs)

    axs[0,i].set_title(f"WY{water_year}") # , fontsize=10
    axs[1,i].set_title(f"")
    
    # make background gray
    axs[0,i].set_facecolor('darkgray')
    axs[1,i].set_facecolor('darkgray')

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_spring_temperature_and_anomaly.png', dpi=300, bbox_inches='tight')

In [ ]:
mean_temp_anomaly_da = sierras_spring_anomaly_full_match_da.rio.clip(sierra_nevada_gdf.geometry).mean(dim=['x','y'])
mean_temp_anomaly_da


In [ ]:
f,ax=plt.subplots(figsize=(12,2),dpi=300)
my_colormap = plt.get_cmap('RdBu_r')
norm = plt.Normalize(vmin=-2, vmax=2)
colors = my_colormap(norm(mean_temp_anomaly_da.values))
mean_temp_anomaly_da.to_series().plot.bar(ax=ax, color=colors, edgecolor='black')
ax.axhline(0, color='black', linestyle='--')
ax.set_ylabel('Average anomaly\n[°C]')
ax.set_xlabel('')

ax.tick_params(axis='x', which='both', bottom=True, top=False, labelbottom=True,length=0)
ax.set_xticklabels([f'WY{water_year}' for water_year in mean_temp_anomaly_da.water_year.values], rotation=0)
ax.set_xticklabels([f'' for water_year in mean_temp_anomaly_da.water_year.values], rotation=0)

# turn off the x axis spines
ax.spines['top'].set_visible(False)

ax.spines['right'].set_visible(True)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)

# label the top of each bar with the value
for i, v in enumerate(mean_temp_anomaly_da.values):
    if v < 0:
        ax.text(i, v - 0.2, f'{v:.1f}', ha='center', va='top')
    else:
        ax.text(i, v + 0.2, f'{v:.1f}', ha='center', va='bottom')#, fontsize=10

ax.set_ylim(-2, 1.7)

# switch the y axis to be on the right

ax.yaxis.tick_right()
ax.yaxis.set_label_position("right")

f.tight_layout()
f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_spring_temp_anomaly_barchart.png', dpi=300, bbox_inches='tight')

In [ ]:
# Initial figure setup
ar = 1.0  # initial aspect ratio for first trial
wi = 10   # width of the whole figure in inches
hi = wi * ar * 0.35  # height factor adjusted for 2 rows

rows, cols = 2, len(sierra_nevada_ds.water_year)   # one column per water year, from the data

f, axs = plt.subplots(rows, cols, 
                      figsize=(wi, hi),
                      subplot_kw={'projection': cartopy_crs},
                      layout='constrained',
                      dpi=300)

# Adjust constrained layout spacing
f.get_layout_engine().set(w_pad=0.00, h_pad=0.00, wspace=0.00, hspace=0.05)

# Get colormaps for text coloring
cmap_runoff = plt.get_cmap('RdBu')
cmap_temp = plt.get_cmap('RdBu_r')

# Plot anomalies for each water year
for i, water_year in enumerate(sierra_nevada_ds.water_year.values):
    
    # Only add colorbar for the last column
    if water_year == sierra_nevada_ds.water_year.values[-1]:  # last column gets the colorbar
        colorbar_flag = True
        cbar_dowy = {"label":"Runoff onset\nanomaly [days]", "ticks":[-30,-15,0,15,30]}
        #cbar_temp = {"label":"Spring temp.\nanomaly [°C]", "ticks":[-3,-2,-1,0,1,2,3]}
        cbar_temp = {"label":"Spring temp.\nanomaly [°C]", "ticks":[-2,-1,0,1,2]}
    else:
        colorbar_flag = False
        cbar_dowy = None
        cbar_temp = None
    
    sierras_hillshade_da.plot.imshow(cmap='grey',ax=axs[0, i],transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)
    sierras_hillshade_da.plot.imshow(cmap='grey',ax=axs[1, i],transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)

    sierra_nevada_ds['runoff_onset_anomaly'].sel(water_year=water_year).plot.imshow(
        ax=axs[0, i], vmin=-30, vmax=30, cmap='RdBu', transform=cartopy_crs,
        add_colorbar=colorbar_flag, cbar_kwargs=cbar_dowy
    )
    sierras_spring_anomaly_full_match_da.sel(water_year=water_year).plot.imshow(
        ax=axs[1, i], vmin=-2, vmax=2, cmap='RdBu_r', transform=cartopy_crs,
        add_colorbar=colorbar_flag, cbar_kwargs=cbar_temp
    )

    axs[0, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
    axs[1, i].gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
    axs[0, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())
    axs[1, i].set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())
    
    # axs[0,i].set_title(f"WY{water_year}")
    axs[0,i].set_title("")
    axs[1,i].set_title("")
    
    axs[0,i].set_aspect('equal')
    axs[1,i].set_aspect('equal')

# Update positions and get proper aspect ratio BEFORE adding text/geometries
plt.draw()

# Get proper ratio from one of the axes
xmin, xmax = axs[0, 0].get_xbound()
ymin, ymax = axs[0, 0].get_ybound()
y2x_ratio = (ymax-ymin) / (xmax-xmin) * rows/cols

# Apply new h/w aspect ratio by changing height
f.set_figheight(wi * y2x_ratio)

# NOW add mountain outlines and text annotations
for i, water_year in enumerate(sierra_nevada_ds.water_year.values):
    # Add mountain outlines
    plot_geoms(sierra_nevada_gdf, axs[0, i], color='black', linewidth=0.8, transform=cartopy_crs)
    plot_geoms(sierra_nevada_gdf, axs[1, i], color='black', linewidth=0.8, transform=cartopy_crs)
    
    # Get mean anomaly values
    runoff_mean_anom = mean_runoff_anomaly_da.sel(water_year=water_year).values
    temp_mean_anom = mean_temp_anomaly_da.sel(water_year=water_year).values
    
    # Format text based on sign
    if runoff_mean_anom < 0:
        runoff_text = f"Mean anomaly:\n{abs(runoff_mean_anom):.1f} days earlier"
    else:
        runoff_text = f"Mean anomaly:\n{runoff_mean_anom:.1f} days later"
    
    if temp_mean_anom < 0:
        temp_text = f"Mean anomaly:\n{abs(temp_mean_anom):.1f} °C cooler"
    else:
        temp_text = f"Mean anomaly:\n{temp_mean_anom:.1f} °C warmer"
        
    # set path colors to black or white based on text color for visibility
    runoff_path_color = 'black' if abs(runoff_mean_anom) < 15 else 'white'
    temp_path_color = 'black' if abs(temp_mean_anom) < 1 else 'white'
    
    # Get colors from colormaps based on normalized values
    runoff_color = cmap_runoff(plt.Normalize(vmin=-30, vmax=30)(runoff_mean_anom))
    temp_color = cmap_temp(plt.Normalize(vmin=-2, vmax=2)(temp_mean_anom))
    
    # Add text in bottom-left corner with black outline
    text1 = axs[0,i].text(0.03, 0.01, runoff_text, 
                        transform=axs[0,i].transAxes, 
                        ha='left', va='bottom', 
                        fontsize=5.8,
                        color=runoff_color,
                        weight='bold')
    text1.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground=runoff_path_color),
                           path_effects.Normal()])
    
    text2 = axs[1,i].text(0.03, 0.01, temp_text, 
                        transform=axs[1,i].transAxes, 
                        ha='left', va='bottom', 
                        fontsize=5.8,
                        color=temp_color,
                        weight='bold')
    text2.set_path_effects([path_effects.Stroke(linewidth=1.5, foreground=temp_path_color),
                           path_effects.Normal()])

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_runoff_onset_and_spring_temp_anomalies.png', 
          dpi=300, bbox_inches='tight', pad_inches=0.01)

plt.close('all')

In [ ]:
# just mock up the hillshade and text to test out different colors
f,ax=plt.subplots(figsize=(2,3),subplot_kw={'projection':cartopy_crs},layout='constrained')
sierras_hillshade_da.plot.imshow(cmap='grey',ax=ax,transform=cartopy_crs,vmin=0,vmax=255,add_colorbar=False)
# add text from previous plot
value = 0
if value < 0:
    text = f"Mean anomaly:\n{abs(value):.1f} °C cooler"
else:
    text = f"Mean anomaly:\n{value:.1f} °C warmer"
temp_color = cmap_temp(plt.Normalize(vmin=-2, vmax=2)(value))
text = f"Mean anomaly:\n{value:.1f} °C warmer"
text2 = ax.text(0.03, 0.01, text, 
                    transform=ax.transAxes, 
                    ha='left', va='bottom', 
                    fontsize=12,
                    color=temp_color,
                    weight='bold')
text2.set_path_effects([path_effects.Stroke(linewidth=2, foreground='white'),
                       path_effects.Normal()])

ax.gridlines(draw_labels=False, linewidth=0.5, color='lightgray', alpha=0.5, linestyle='--')
ax.set_extent([-122.2, -117.8, 34.5, 40.7], crs=ccrs.PlateCarree())

ax.set_title("")

ax.set_aspect('equal')


In [ ]:
f,ax=plt.subplots(figsize=(4,4),dpi=300)

combined_df = pd.DataFrame({
    'era5_anom': mean_temp_anomaly_da.values,
    'melt_anom': mean_runoff_anomaly_da.values,
    'water_year': mean_runoff_anomaly_da.water_year.values
}).dropna()

n = len(combined_df)

# plot temperature anomalies on x axis, melt anomalies on y axis,color should be water year
plot = ax.scatter(combined_df['era5_anom'],combined_df['melt_anom'],c=combined_df['water_year'],cmap='YlGnBu',vmin=config.water_years[0],vmax=config.water_years[-1], edgecolors='black',s=50,zorder=2)

# vertical and horizontal dashed lines at 0
ax.axhline(0, color='black', linestyle=':')
ax.axvline(0, color='black', linestyle=':')

# ax.set_xlabel('10-year (WY2015-WY2024) Spring Temperature Anomaly [°C]')
# ax.set_ylabel('10-year (WY2015-WY2024) Snowmelt Runoff Onset Anomaly [days]')

ax.set_xlabel(f'Average spring 2m temperature anomaly [°C]', fontsize=9)
ax.set_ylabel('Average runoff onset anomaly [days]',fontsize=9)

ax.set_xlim([-3.2,3.2])
ax.set_ylim([-42,42])

# create a grid
ax.grid(True, which='both', linestyle='--', linewidth=0.5, zorder=0)


# turn off the frame
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.set_xticks([-3,-2,-1,0,1,2,3])
ax.set_yticks([-40,-30,-20,-10,0,10,20,30,40])

slope, intercept = np.polyfit(combined_df['era5_anom'], combined_df['melt_anom'], 1)

corr = np.corrcoef(combined_df['era5_anom'], combined_df['melt_anom'])[0, 1]

title = 'Sierra Nevada'
if len(title)>40:
    title = "\n".join(textwrap.wrap(title, 40))

#ax.set_title(f'{title}\nn: {len(combined_df)} | r: {corr:.2f} | slope: {slope:.1f} days/°C')
ax.set_title('')

stats_text = f'n = {n}\nr = {corr:.2f}\nslope = {slope:.1f} days/°C'
ax.text(0.97, 0.97, stats_text, 
        transform=ax.transAxes,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
        #fontsize=9,
        )

x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
expected_melt_anom = slope * x_smooth + intercept
ax.plot(x_smooth, expected_melt_anom, color='red', linestyle='--',zorder=3)

# now add a legend for the year colors (legend, not a colorbar)
# cbar = f.colorbar(plot, ax=ax, orientation='vertical', pad=0.02)
# cbar.set_label('Water Year', rotation=270, labelpad=15)
# cbar.set_ticks(np.arange(2015, 2025, 1))
# cbar.set_ticklabels([str(year) for year in range(2015, 2025)])

from matplotlib.lines import Line2D
cmap = plt.get_cmap('YlGnBu')
norm = plt.Normalize(vmin=config.water_years[0], vmax=config.water_years[-1])
legend_elements = []
for year in config.water_years:
    color = cmap(norm(year))
    legend_elements.append(Line2D([0], [0], marker='o', color='w', 
                                   markerfacecolor=color, markeredgecolor='black',
                                   markersize=7, label=f'WY{year}'))

# Add legend outside to the right
ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(0.725, 0.5), edgecolor='black',
          framealpha=0.8, fontsize=7)# cmap = plt.get_cmap('YlGnBu')

f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_spring_temp_vs_runoff_onset_anomaly_scatter.png', dpi=300, bbox_inches='tight',pad_inches=0)

In [ ]:
# create freestanding legend for year colors without using any data (cmap YlGnBu over config.water_years)
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
# import Line2D
f, ax = plt.subplots(figsize=(2,4), dpi=300)
cmap = plt.get_cmap('YlGnBu')
norm = plt.Normalize(vmin=config.water_years[0], vmax=config.water_years[-1])
legend_elements = []
for year in config.water_years:
    color = cmap(norm(year))
    legend_elements.append(mpl.lines.Line2D([0], [0], marker='o', color='w', 
                                   markerfacecolor=color, markeredgecolor='black',
                                   markersize=7, label=f'WY{year}'))
ax.legend(handles=legend_elements, loc='center', edgecolor='black',
          framealpha=0.8, fontsize=8)
ax.axis('off')
f.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_spring_temp_vs_runoff_onset_anomaly_legend.png', dpi=300, bbox_inches='tight',pad_inches=0.01)

## Worked-example composite

One figure: the N-year medians (left), the per-year anomaly maps of runoff onset (top) and spring
2 m temperature (bottom) with their range means, and the temperature-sensitivity scatter (right).
Everything above must have run (FCF ≤ 50 % applies). Added 2026-09-01 — not yet executed against a
store; adjust `figsize`/`width_ratios` after the first render.

In [ ]:
years = sierra_nevada_ds.water_year.values
n_years = len(years)
extent = [-122.2, -117.8, 34.5, 40.7]
cmap_runoff, cmap_temp = plt.get_cmap('RdBu'), plt.get_cmap('RdBu_r')

fig = plt.figure(figsize=(18, 5.4), dpi=300)
gs = fig.add_gridspec(2, n_years + 4,
                      width_ratios=[1.0] + [1.0] * n_years + [0.08] + [0.55] + [2.8],
                      wspace=0.04, hspace=0.06, left=0.03, right=0.99, top=0.82, bottom=0.04)


def map_axes(cell):
    ax = fig.add_subplot(cell, projection=cartopy_crs)
    sierras_hillshade_da.plot.imshow(cmap='grey', ax=ax, transform=cartopy_crs, vmin=0, vmax=255,
                                     add_colorbar=False, zorder=0)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.4, color='lightgray', alpha=0.5, linestyle='--')
    ax.set_title('')
    ax.set_aspect('equal')
    plot_geoms(sierra_nevada_gdf, ax, color='black', linewidth=0.6, transform=cartopy_crs, zorder=3)
    return ax


# left column: the N-year medians, each with a small horizontal colorbar underneath
ax_med = map_axes(gs[0, 0])
im = sierra_nevada_ds['runoff_onset_median'].plot.imshow(ax=ax_med, vmin=110, vmax=250, cmap='viridis',
                                                         transform=cartopy_crs, add_colorbar=False, zorder=1)
fig.colorbar(im, cax=ax_med.inset_axes([0.08, -0.09, 0.84, 0.05]), orientation='horizontal',
             ticks=[110, 180, 250]).ax.tick_params(labelsize=5, length=2, pad=1)
ax_med.set_title(f'WY{years[0]}-{years[-1]} median', fontsize=8, fontweight='bold', color='tab:purple')
ax_tmed = map_axes(gs[1, 0])
im = (sierras_spring_temp_10yr_median_da - 273.15).plot.imshow(ax=ax_tmed, vmin=-5, vmax=10, cmap='plasma',
                                                                transform=cartopy_crs, add_colorbar=False, zorder=1)
ax_tmed.set_title('')
fig.colorbar(im, cax=ax_tmed.inset_axes([0.08, -0.09, 0.84, 0.05]), orientation='horizontal',
             ticks=[-5, 0, 5, 10]).ax.tick_params(labelsize=5, length=2, pad=1)

# middle columns: one water year each, runoff onset anomaly on top, spring temperature anomaly below
first_anom_ax = None
for i, wy in enumerate(years):
    ax_a = map_axes(gs[0, i + 1])
    ax_t = map_axes(gs[1, i + 1])
    first_anom_ax = first_anom_ax or ax_a
    im_a = sierra_nevada_ds['runoff_onset_anomaly'].sel(water_year=wy).plot.imshow(
        ax=ax_a, vmin=-30, vmax=30, cmap='RdBu', transform=cartopy_crs, add_colorbar=False, zorder=1)
    im_t = sierras_spring_anomaly_full_match_da.sel(water_year=wy).plot.imshow(
        ax=ax_t, vmin=-2, vmax=2, cmap='RdBu_r', transform=cartopy_crs, add_colorbar=False, zorder=1)
    ax_a.set_title(f'WY{wy}', fontsize=8)
    ax_t.set_title('')                      # xarray's automatic coordinate title
    for ax, val, cmap, lim, neg, pos in [
            (ax_a, float(mean_runoff_anomaly_da.sel(water_year=wy)), cmap_runoff, 30, 'days earlier', 'days later'),
            (ax_t, float(mean_temp_anomaly_da.sel(water_year=wy)), cmap_temp, 2, '°C cooler', '°C warmer')]:
        txt = f"Mean anomaly:\n{abs(val):.1f} {neg if val < 0 else pos}"
        t = ax.text(0.03, 0.01, txt, transform=ax.transAxes, ha='left', va='bottom', fontsize=5,
                    color=cmap(plt.Normalize(-lim, lim)(val)), weight='bold', zorder=4)
        t.set_path_effects([path_effects.Stroke(linewidth=1.2, foreground='black' if abs(val) < lim / 2 else 'white'),
                            path_effects.Normal()])
last_anom_ax = ax_a

# vertical colorbars for the two anomaly rows
cb_a = fig.colorbar(im_a, cax=fig.add_subplot(gs[0, n_years + 1]), ticks=[-30, -15, 0, 15, 30])
cb_a.set_label('Runoff onset\nanomaly [days]', fontsize=7); cb_a.ax.tick_params(labelsize=6)
cb_t = fig.colorbar(im_t, cax=fig.add_subplot(gs[1, n_years + 1]), ticks=[-2, -1, 0, 1, 2])
cb_t.set_label('Spring temp.\nanomaly [°C]', fontsize=7); cb_t.ax.tick_params(labelsize=6)

# right: the temperature-sensitivity scatter (same construction as the standalone scatter cell)
ax_sc = fig.add_subplot(gs[:, n_years + 3])
combined_df = pd.DataFrame({'era5_anom': mean_temp_anomaly_da.values, 'melt_anom': mean_runoff_anomaly_da.values,
                            'water_year': mean_runoff_anomaly_da.water_year.values}).dropna()
ax_sc.scatter(combined_df['era5_anom'], combined_df['melt_anom'], c=combined_df['water_year'], cmap='YlGnBu',
              vmin=years[0], vmax=years[-1], edgecolors='black', s=45, zorder=2)
ax_sc.axhline(0, color='black', linestyle=':'); ax_sc.axvline(0, color='black', linestyle=':')
ax_sc.set_xlim([-3.2, 3.2]); ax_sc.set_ylim([-42, 42])
ax_sc.set_xticks([-3, -2, -1, 0, 1, 2, 3]); ax_sc.set_yticks([-40, -30, -20, -10, 0, 10, 20, 30, 40])
ax_sc.grid(True, which='both', linestyle='--', linewidth=0.5, zorder=0)
for s in ax_sc.spines.values():
    s.set_visible(False)
ax_sc.tick_params(axis='both', pad=0, length=0, labelsize=7)
ax_sc.set_xlabel('Average spring 2m temperature anomaly [°C]', fontsize=8)
ax_sc.set_ylabel('Average runoff onset anomaly [days]', fontsize=8)
slope, intercept = np.polyfit(combined_df['era5_anom'], combined_df['melt_anom'], 1)
corr = np.corrcoef(combined_df['era5_anom'], combined_df['melt_anom'])[0, 1]
x_smooth = np.linspace(combined_df['era5_anom'].min(), combined_df['era5_anom'].max(), 20)
ax_sc.plot(x_smooth, slope * x_smooth + intercept, color='red', linestyle='--', zorder=3)
ax_sc.text(0.97, 0.97, f'n = {len(combined_df)}\nr = {corr:.2f}\nslope = {slope:.1f} days/°C',
           transform=ax_sc.transAxes, va='top', ha='right', fontsize=7,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
from matplotlib.lines import Line2D
norm_year = plt.Normalize(vmin=years[0], vmax=years[-1])
ax_sc.legend(handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor=plt.get_cmap('YlGnBu')(norm_year(y)),
                             markeredgecolor='black', markersize=5, label=f'WY{y}') for y in years],
             loc='lower left', fontsize=5.5, edgecolor='black', framealpha=0.8)
ax_sc.set_title('"Temperature sensitivity" scatter plot', fontsize=9)
direction = 'earlier' if slope < 0 else 'later'
ax_sc.text(0.0, -0.14, f'Interpretation\nIn the Sierra Nevada (WY{years[0]}-{years[-1]}), for every 1°C warmer average '
           f'spring\nair temperature, runoff onset occurs {abs(slope):.1f} days {direction}.',
           transform=ax_sc.transAxes, va='top', ha='left', fontsize=7, style='italic')

# headers and row labels (positions from the drawn axes)
fig.canvas.draw()
x0, x1 = first_anom_ax.get_position().x0, last_anom_ax.get_position().x1
fig.text((x0 + x1) / 2, 0.895, f'annual anomaly (annual − WY{years[0]}-{years[-1]} median)', ha='center', fontsize=9)
fig.text(0.006, ax_med.get_position().y0 + ax_med.get_position().height / 2, 'Runoff onset',
         rotation=90, va='center', ha='center', fontsize=9)
fig.text(0.006, ax_tmed.get_position().y0 + ax_tmed.get_position().height / 2, 'ERA5-Land spring\n2m temp.',
         rotation=90, va='center', ha='center', fontsize=9)
fig.suptitle(f'{n_years}-yr runoff onset anomaly vs {n_years}-yr spring temperature anomaly in Sierra Nevada, California, USA',
             fontsize=12, y=0.97)

fig.savefig(paths.figdir('case_studies/sierra_nevada', config.version) / 'sierra_nevada_worked_example.png',
            dpi=300, bbox_inches='tight', pad_inches=0.05)